# Founder-Departure GitHub Commit Corpus — Demo\n\nThis notebook demonstrates the packaging pipeline behind the **Founder-Departure GitHub Commit Corpus** dataset artifact.\n\nThe original pipeline (not re-run here — it requires a `GH_TOKEN`, `git clone --bare` on 1,170 candidate repos, and multiple GB of scratch data) does the following:\n\n1. Stratified-sample GitHub repos across 6 languages x 3 star strata (18 cells) via the GitHub Search API.\n2. Clone each candidate `--bare` and extract full commit history with `git log --numstat`.\n3. Apply a *bulk-import-artifact* filter (Kalliamvakou et al.) and a *single-dominant-founder* filter to keep only repos that plausibly started with one founder.\n4. Tag each non-founder commit with a `diffusion_window_tag` relative to an approximate founder Truck-Factor/DOA-departure (TFDD) point, and each contributor with their `contributor_tenure_days` (write-access-duration proxy).\n5. Package everything into the `exp_sel_data_out` schema: one example per (commit, file) row, `output` = `is_founder_commit` label, `input` = the JSON-serialized row with author identity withheld (to prevent label leakage).\n\n**This demo notebook re-runs the actual packaging code** (`data.py`'s `stride_cap` / `to_example` / `main` logic) on a small curated subset of already-crawled rows (`mini_demo_data.json`, 104 rows across 8 repos), so you can see the transformation from raw per-(commit,file) rows to the final dataset schema without needing a GitHub token or a multi-hour crawl.

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# The original data.py has zero third-party dependencies (stdlib json/collections/pathlib only).\n# This notebook additionally uses matplotlib for the results visualization cell, which is\n# pre-installed on Colab, so it only needs installing locally to match Colab's exact version.\nif 'google.colab' not in sys.modules:\n    _pip('matplotlib==3.10.0')

In [ ]:
# Original imports from data.py, plus matplotlib for the visualization cell at the end.\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt

## Load the demo data\n\n`mini_demo_data.json` is a curated subset of the pipeline's intermediate output: 104 raw per-(commit,file) rows drawn from 8 of the 254 final repos (the point in the original pipeline right before `data.py` packages them, i.e. equivalent to what would live in `temp/datasets/github_founder_corpus_rows.jsonl` plus the `funnel_report.json` summary). It is loaded from GitHub with a local-file fallback so this notebook works both on Colab and locally.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-24ffbe-pre-departure-bus-factor-diffusion/main/round-2/dataset-1/demo/mini_demo_data.json"\nimport json, os\n\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception: pass\n    if os.path.exists(\"mini_demo_data.json\"):\n        with open(\"mini_demo_data.json\") as f: return json.load(f)\n    raise FileNotFoundError(\"Could not load mini_demo_data.json\")

In [ ]:
data = load_data()\nraw_rows = data[\"rows\"]\nfunnel = data[\"funnel\"]\nprint(f\"loaded {len(raw_rows)} raw (commit,file) rows\")\nprint(f\"{len(set(r['full_name'] for r in raw_rows))} distinct repos\")

## Config\n\n`MAX_ROWS_PER_REPO` is the original pipeline's per-repo chronological-stride cap (200 rows/repo in the full run, used to bound corpus size to under the 100MB limit). Since the demo repos only have ~13 rows each, we set a small cap here so `stride_cap` still exercises its striding logic — bump it toward 200 to match the original run exactly (it will simply be a no-op on this small subset once it exceeds each repo's row count).

In [ ]:
# MAX_ROWS_PER_REPO = 200  # original full-run value\nMAX_ROWS_PER_REPO = 5  # small demo value: small enough that stride_cap actually strides on this subset\n\nLABEL_FIELD = \"is_founder_commit\"\nINPUT_FIELDS_EXCLUDE = {LABEL_FIELD, \"author_alias_key\", \"author_email\", \"author_name\"}

## Group rows by repo\n\nSame as `load_rows_by_repo()` in `data.py`, except the original reads a JSONL file line-by-line; here the equivalent rows are already loaded in memory as `raw_rows`.

In [ ]:
def load_rows_by_repo(rows):
    by_repo = defaultdict(list)
    for row in rows:
        by_repo[row["full_name"]].append(row)
    return by_repo


by_repo = load_rows_by_repo(raw_rows)
for full_name, rows in by_repo.items():
    print(f"{full_name}: {len(rows)} rows")

## Chronological-stride cap and example packaging\n\n`stride_cap` is copied verbatim from `data.py`: it strides evenly through a repo's chronologically-sorted rows to keep at most `cap` of them, rather than truncating (which would bias toward early history). `to_example` is also copied verbatim: it builds the `exp_sel_data_out` schema example, JSON-serializing every row field except the label and the withheld author-identity fields into `input`, and setting `output` to the founder-vs-other label.

In [ ]:
def stride_cap(rows, cap):
    n = len(rows)
    if n <= cap:
        return rows
    step = n / cap
    return [rows[int(i * step)] for i in range(cap)]


def to_example(row):
    input_obj = {k: v for k, v in row.items() if k not in INPUT_FIELDS_EXCLUDE}
    example = {
        "input": json.dumps(input_obj, sort_keys=True),
        "output": str(row[LABEL_FIELD]),
        "metadata_fold": row["search_lang_query"] + "|" + row["search_stars_bucket"],
        "metadata_task_type": "classification",
        "metadata_n_classes": 2,
        "metadata_full_name": row["full_name"],
        "metadata_primary_language": row["primary_language"],
        "metadata_search_stars_bucket": row["search_stars_bucket"],
        "metadata_commit_sha": row["commit_sha"],
        "metadata_commit_timestamp": row["commit_timestamp"],
        "metadata_commit_index": row["commit_index"],
        "metadata_n_commits_total": row["n_commits_total"],
        "metadata_contributor_tenure_days": row["contributor_tenure_days"],
        "metadata_founder_tfdd_approx": row["founder_tfdd_approx"],
        "metadata_diffusion_window_tag": row["diffusion_window_tag"],
        "metadata_alias_ambiguous_repo": row["alias_ambiguous_repo"],
    }
    return example

## Assemble the packaged dataset\n\nThis mirrors `main()` in `data.py`: for each repo, sort rows chronologically by `commit_index`, stride-cap them, convert each to an `exp_sel_data_out` example, and assemble the final `datasets` + `metadata` structure (the `metadata.funnel` block here is the same funnel report as the full run, since the funnel is computed once over all 1,170 candidates and is independent of which repos this demo subset draws from).

In [ ]:
examples = []
for full_name, rows in by_repo.items():
    rows_sorted = sorted(rows, key=lambda r: r["commit_index"])
    kept = stride_cap(rows_sorted, MAX_ROWS_PER_REPO)
    for row in kept:
        examples.append(to_example(row))

description = (
    "Per-(commit,file) rows for 254 GitHub repos passing a fame-independent "
    "stratified sample (6 languages x 3 star strata, 1170 candidates -> 254 "
    "final, full funnel in metadata) and founder-only-start filters (>=1095 "
    "days history, <=80% of files touched in first 7 days, single author "
    ">=70% of commits in first 6mo/50 commits). `output` is founder-vs-other "
    "authorship of that (commit,file) row; `input` withholds author identity "
    "to prevent label leakage for downstream DOA/classification use."
    f" [DEMO SUBSET: {len(examples)} examples from {len(by_repo)} repos, "
    f"MAX_ROWS_PER_REPO={MAX_ROWS_PER_REPO}]"
)

out = {
    "datasets": [
        {
            "dataset": "github_founder_departure_corpus",
            "examples": examples,
        }
    ],
    "metadata": {
        "source": (
            "GitHub REST search/repositories API (candidate discovery, "
            "GH_TOKEN-authenticated) + local `git clone --bare` / "
            "`git log --numstat` (full commit history extraction, avoids "
            "API rate limits)."
        ),
        "description": description,
        "n_examples": len(examples),
        "n_repos": len(by_repo),
        "funnel": funnel,
    },
}

print(f"packaged {len(examples)} examples across {len(by_repo)} repos")

## Results\n\nA quick look at the packaged examples: the founder-vs-other label balance in this demo subset, and the full-run filtering funnel (from `metadata.funnel`, computed over all 1,170 sampled candidates in the original run) showing how many repos survived each filter per language.

In [ ]:
from collections import Counter

label_counts = Counter(ex["output"] for ex in examples)
print("Label balance (this demo subset):")
for label, count in sorted(label_counts.items()):
    tag = "founder commit" if label == "1" else "non-founder commit"
    print(f"  output={label} ({tag}): {count} examples")

fold_counts = Counter(ex["metadata_fold"] for ex in examples)
print("\nExamples per language|star-stratum fold (this demo subset):")
for fold, count in sorted(fold_counts.items()):
    print(f"  {fold}: {count}")

# Full-run funnel: repos surviving each stage, aggregated per language across all 3 star strata.
by_cell = funnel["by_cell"]
langs = sorted({k.split("|")[0] for k in by_cell})
sampled = [sum(by_cell[f"{l}|{s}"]["sampled"] for s in ["50-500", "500-5000", "5000-100000"]) for l in langs]
final = [sum(by_cell[f"{l}|{s}"]["final_processed"] for s in ["50-500", "500-5000", "5000-100000"]) for l in langs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(["non-founder (0)", "founder (1)"], [label_counts.get("0", 0), label_counts.get("1", 0)], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Demo subset: commit-row label balance")
axes[0].set_ylabel("count")

x = range(len(langs))
axes[1].bar(x, sampled, width=0.4, label="sampled", color="#999999", align="edge")
axes[1].bar([i + 0.4 for i in x], final, width=0.4, label="final_processed", color="#55A868", align="edge")
axes[1].set_xticks([i + 0.4 for i in x])
axes[1].set_xticklabels(langs, rotation=30, ha="right")
axes[1].set_title("Full-run funnel: repos sampled vs. surviving all filters")
axes[1].set_ylabel("repo count")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nFull-run totals: {funnel['totals']}")